<a href="https://colab.research.google.com/github/Song-yiJung/korean-modern-document-ocr/blob/main/01_step1_vision/step1_vision_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Step 1: Google Cloud Vision OCR**

## 매뉴얼 챕터 3.1 대응

* 이 파일은 Google Colab에서 *셀 단위로* 복사·실행하는 것을 전제로 한다.
* 각 셀 사이 구분선(# %% ...)은 Jupytext 형식. Colab/Jupyter에서 자동으로 셀로 분리된다.
* 본인 자료에 맞게 *수정해야 할 곳은 단 한 셀*(아래 ★ 사용자 설정). 그 외는 모두 그대로 두면 된다.
* 이 노트북은 사료 이미지를 한 장씩 Google Cloud Vision API에 보내 1차 텍스트를 추출하고, 폴더 단위로 일괄 처리한다.

**준비:**
1. Google Cloud Vision API 키 파일(`.json`) — 부록 A.2 절차로 발급
2. 사료 이미지가 든 폴더 — Google Drive 안에 둠
3. 결과를 저장할 폴더 — Google Drive 안에 (자동 생성됨)

**예상 시간·비용 (참고):**
- 처리 시간: 사료 한 장당 약 1~3초
- 비용: 월 1,000회까지 무료

## **시작 전 준비**

### 1. 이 노트북을 본인 계정에 복사

상단 메뉴에서 **`파일 → Drive에 사본 저장`** 을 클릭한다.
원본은 읽기 전용이므로 *반드시* 본인 계정의 사본으로 작업한다.
복사된 노트북은 본인 Google Drive의 `Colab Notebooks` 폴더에
자동 저장된다. 이후로는 그 사본을 열어서 사용하면 된다.

#### ① 필요한 패키지 설치

코랩은 매 세션마다 환경이 초기화되므로 *매번* 실행한다.

`!pip` 명령 앞의 느낌표는 "셸 명령으로 실행하라"는 뜻이다.

In [ ]:
!pip install -q google-cloud-vision

### ② Google Drive 마운트

사료 이미지를 읽고 처리 결과를 저장하려면 Drive를 코랩에 연결해야
한다. 실행하면 권한 요청 창이 뜬다. 본인 구글 계정으로 인증.

마운트가 완료되면 코랩의 `/content/drive/MyDrive/` 경로로 본인
Drive의 최상위(MyDrive)에 접근할 수 있다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### ③ ★ 사용자 설정 — 본인 환경에 맞게 *여기만 수정*

이 셀이 본 노트북에서 **본인이 수정해야 할 유일한 셀**이다.
다른 셀은 그대로 두고 위에서 아래로 실행만 하면 된다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용자 설정 (총 4개)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━

####[1] ision API 키 파일 경로를 등록한 Colab Secret의 *이름*.

▸ 등록 절차:

좌측 사이드바의 🔑(열쇠) 아이콘 클릭 → `새 보안 비밀` 추가
 - 이름: VISION_KEY_PATH   (아래 변수와 일치해야 함)
 - 값:  Drive 안 .json 파일의 코랩 경로
     
     예) /content/drive/MyDrive/keys/vision_key.json
 - `노트북 액세스` 토글 ON

▸ 키 *내용*이 아니라 *경로*를 저장하는 이유:

 - Drive에 원본 한 부만 두고 step2 노트북에서도 공유 사용한다.
 - 매번 키 파일을 새로 업로드할 필요가 없다.
VISION_KEY_SECRET_NAME = 'VISION_KEY_PATH'

####[2] 사료 이미지가 있는 *Drive 폴더* 경로.
- MyDrive/ 다음에 본인이 만든 폴더 이름을 적는다.
- 하위 폴더 안 이미지까지 *전부 자동 수집*된다.

    INPUT_DIR  = '/content/drive/MyDrive/사료_이미지'

####[3] 결과 저장 폴더 (없으면 자동 생성).
* OUTPUT_DIR = '/content/drive/MyDrive/vision_결과'

####[4] 사료의 주요 언어 힌트.
* 모델에 "이 언어를 우선 판독하라"고 알리는 *권장 신호*.
* 일본어/구자체 한자 사료 (1900~1945)  →  ['ja', 'zh-Hant']
* 한글 문서                          →  ['ko', 'en']
* 한문 (전통 한자)                    →  ['zh-Hant', 'zh-Hans']
* 영문 중심                          →  ['en']
* 다국어 혼용은 2~3개까지 권장 (너무 많이 넣으면 오히려 혼란).
LANGUAGE_HINTS = ['ja', 'zh-Hant']

#

### ④ Vision API 인증 등록 + 폴더 점검

`GOOGLE_APPLICATION_CREDENTIALS`는 Google Cloud 라이브러리가
인증 정보를 찾는 *표준 환경변수*. 이 줄을 실행한 후 Vision 클라이언트를
생성하면 자동으로 인증이 처리된다.

만약 *Secret 액세스 안내 창*이 뜨면 노트북 액세스 토글을 켠다.

In [ ]:
import os
from pathlib import Path
from google.colab import userdata

# 1) Secret에서 키 파일 경로 읽기
try:
    key_path = userdata.get(VISION_KEY_SECRET_NAME)
except Exception as e:
    raise RuntimeError(
        f"Colab Secret '{VISION_KEY_SECRET_NAME}' 접근 실패: {e}\n"
        "→ 좌측 🔑 아이콘에서 (1) 시크릿이 등록되어 있는지, "
        "(2) 이름이 정확한지, (3) '노트북 액세스' 토글이 켜져 있는지 확인하세요."
    )

# 2) 환경변수 등록
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = key_path
print(f"✔ Vision API 인증 정보 등록 완료")
print(f"  키 파일: {key_path}")

# 3) 입출력 폴더 검증
INPUT_DIR  = Path(INPUT_DIR)
OUTPUT_DIR = Path(OUTPUT_DIR)

if not INPUT_DIR.exists():
    raise RuntimeError(
        f"입력 폴더가 존재하지 않습니다: {INPUT_DIR}\n"
        "→ 셀 ③의 INPUT_DIR 경로가 맞는지, "
        "Drive에 그 이름의 폴더를 실제로 만들었는지 확인하세요."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"  입력 폴더: {INPUT_DIR}")
print(f"  출력 폴더: {OUTPUT_DIR}")

### ⑤ Vision 호출 함수 정의

`document_text_detection`은 사료·문서처럼 글자 밀도가 높고 행·단락
구조가 명확한 입력에 최적화되어 있다. (단문 표지판·영수증은
`text_detection`을 쓴다.)

이 함수는 두 가지를 반환한다:

- `text` — 평탄화된 본문 텍스트 (검색·후속 교정 입력용)
- `full` — 글자별 좌표·신뢰도·언어 감지 등 *구조 정보*
  (디지털 판본 단계에서 활용 — 지금은 함께 저장만)

In [ ]:
from google.cloud import vision
from google.protobuf import json_format

vision_client = vision.ImageAnnotatorClient()


def extract_text_from_image(img_path: Path):
    """이미지 한 장을 Vision API로 처리 → (텍스트, 구조 dict) 반환."""
    with open(img_path, "rb") as f:
        content = f.read()

    image = vision.Image(content=content)
    image_context = vision.ImageContext(language_hints=LANGUAGE_HINTS)

    response = vision_client.document_text_detection(
        image=image,
        image_context=image_context,
    )

    if response.error.message:
        raise Exception(f"Vision API 통신 오류: {response.error.message}")

    if response.full_text_annotation:
        text = response.full_text_annotation.text.strip()
        full = json_format.MessageToDict(response.full_text_annotation._pb)
        return text, full
    return "", {}


print("✔ Vision 호출 함수 정의 완료")

### ⑥ 처리 대상 이미지 수집

폴더 내 `.jpg`/`.jpeg`/`.png`/`.tif`/`.tiff` 확장자를 모두 수집한다.
`rglob`이라 하위 폴더까지 *재귀적*으로 포함 — 권/문건 폴더로
정리된 사료도 그대로 작동한다.

이미지가 한 장도 없다면 셀 ③의 `INPUT_DIR` 경로가 맞는지,
이미지를 실제로 그 폴더에 올렸는지 확인한다.

In [ ]:
valid_extensions = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}
image_paths = sorted([
    p for p in INPUT_DIR.rglob('*')
    if p.suffix.lower() in valid_extensions
])

print(f"처리 대상: 총 {len(image_paths)}장")
if image_paths:
    print("처음 3장:")
    for p in image_paths[:3]:
        print(f"  {p.relative_to(INPUT_DIR)}")
else:
    print("⚠️ 이미지가 한 장도 없습니다.")
    print("   → INPUT_DIR 경로와 업로드 여부를 확인하세요.")

### ⑦ 일괄 처리

### 핵심 로직

- **중복 처리 방지(멱등성)**: 결과 JSON이 이미 있으면 건너뜀.
  → 중단했다가 다시 실행해도 처음부터 안 돌린다.
- **에러 분리 기록**: 일시 오류는 `.err.json`으로 따로 기록.
  다음 실행 시 *자동 재시도*.

### 출력 폴더 구조

```
결과 폴더/
  ├─ <파일ID>.json       ← 평탄 텍스트 + 좌표 구조 + 메타
  ├─ <파일ID>.txt        ← 평탄 텍스트만 (사람이 읽기 쉬움)
  └─ <파일ID>.err.json   ← (에러 시) 다음 실행 때 자동 재시도
```

`.json`은 step2의 입력, `.txt`는 사람이 빠르게 훑어보기 용이다.

In [ ]:
import json
from datetime import datetime

processed = skipped = errors = 0

for idx, img_path in enumerate(image_paths, 1):
    rel_path = img_path.relative_to(INPUT_DIR)
    out_folder = OUTPUT_DIR / rel_path.parent
    out_folder.mkdir(parents=True, exist_ok=True)

    base = img_path.stem
    json_path = out_folder / f"{base}.json"
    err_path  = out_folder / f"{base}.err.json"
    txt_path  = out_folder / f"{base}.txt"

    # 멱등성: 정상 JSON이 이미 있으면 스킵
    if json_path.exists():
        skipped += 1
        continue

    print(f"[{idx}/{len(image_paths)}] {rel_path}", flush=True)
    try:
        text, full = extract_text_from_image(img_path)

        result = {
            "file_path": str(rel_path),
            "vision_raw": text,
            "vision_full": full,            # 좌표·신뢰도 보존 (향후 활용)
            "processed_at": datetime.now().isoformat(timespec='seconds'),
            "gemini_corrected": ""          # step2가 채울 자리
        }
        json_path.write_text(
            json.dumps(result, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )

        # 과거 에러 기록 정리 (이번에 성공했으므로)
        if err_path.exists():
            err_path.unlink()

        if text:
            txt_path.write_text(text, encoding="utf-8")
            print(f"   -> {len(text)}자 추출")
        else:
            print(f"   -> 텍스트 없음 (도면·사진·공백 페이지 가능)")
        processed += 1

    except Exception as e:
        err = {
            "file_path": str(rel_path),
            "error": str(e),
            "failed_at": datetime.now().isoformat(timespec='seconds')
        }
        err_path.write_text(
            json.dumps(err, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )
        print(f"   -> 오류: {str(e)[:100]}")
        errors += 1

print(f"\n완료. 처리 {processed} / 스킵 {skipped} / 오류 {errors}")


### ⑧ 결과 점검

다음 사항을 빠르게 확인:

- 입력 파일 수 ↔ 결과 파일 수 일치 여부
- 글자 50자 미만 페이지 (이미지 품질 문제 신호 —
  도면·사진·공백은 정상)
- 에러 JSON 유무 (있으면 셀 ⑦ 재실행으로 자동 재시도)

In [ ]:
all_jsons = list(OUTPUT_DIR.rglob('*.json'))
ok_jsons  = [f for f in all_jsons if not f.name.endswith('.err.json')]
err_jsons = [f for f in all_jsons if f.name.endswith('.err.json')]

print(f"정상 결과: {len(ok_jsons)} / 입력: {len(image_paths)}")
print(f"에러 기록: {len(err_jsons)}건  (있으면 셀 ⑦ 다시 실행하면 자동 재시도)")

print("\n--- 글자 50자 미만 페이지 점검 ---")
short_ones = []
for jp in ok_jsons:
    data = json.loads(jp.read_text(encoding='utf-8'))
    n = len(data.get('vision_raw', ''))
    if n < 50:
        short_ones.append((jp.stem, n))

if not short_ones:
    print("  없음.")
else:
    for fid, n in short_ones:
        print(f"  {fid}: {n}자")
print(f"\n총 {len(short_ones)}건")

### ⑨ 다음 단계

여기까지 완료하면 사료 한 장당 한 개의 JSON이 `OUTPUT_DIR`에
생성된다.

각 JSON의 핵심 필드:

- `vision_raw` — 평탄 텍스트 (step2의 입력)
- `vision_full` — 글자별 좌표·신뢰도 (디지털 판본 단계 활용)
- `gemini_corrected` — 빈 값 (step2가 채움)

이 결과를 입력으로 받아 **문맥 기반 교정**을 수행하는 절차가
다음 노트북 `step2_gemini_colab.py` 에서 이어진다.